# 03 - Análise de Segurança: Hotspots e Nota por Bairro

Agrega as ocorrências por bairro, calcula um índice de risco, gera visualizações e salva a entrega final da dimensão de segurança.

In [ ]:
from pathlib import Path

import folium
import pandas as pd
import plotly.express as px
from folium.plugins import HeatMap
from IPython.display import HTML

In [ ]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()


ROOT = find_root()
YEAR = 2025
CSV_SEPARATOR = ','


def read_csv_flex(path):
    return pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig')


INPUT_PATH = ROOT / 'data' / 'processed' / 'ocorrencias_seguranca_recife.csv'
BAIRROS_REFERENCIA_PATH = ROOT / 'data' / 'processed' / 'notas_renda.csv'
OUTPUT_BAIRROS = ROOT / 'data' / 'processed' / 'recife_seguranca_bairros.csv'
OUTPUT_NOTAS = ROOT / 'data' / 'processed' / 'notas_seguranca.csv'
OUTPUT_MAPA = ROOT / 'docs' / 'mapa_hotspots_seguranca.html'
OUTPUT_SDS = ROOT / 'data' / 'processed' / 'seguranca_contexto_sds_recife_2025.csv'

print(f'Entrada: {INPUT_PATH}')

In [ ]:
# Carrega a base limpa e os bairros de referência do projeto
df = read_csv_flex(INPUT_PATH)
bairros_referencia = read_csv_flex(BAIRROS_REFERENCIA_PATH)['bairro']

for col in ['latitude', 'longitude', 'mortos', 'feridos', 'baleados']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Ocorrências limpas: {len(df)}')
print(f'Bairros com ocorrência: {df["bairro"].nunique()}')
print(f'Bairros na referência: {len(bairros_referencia)}')
display(df.head(10).reset_index(drop=True).style.hide(axis='index'))

## 1 - Agregação por bairro

In [ ]:
agg = (
    df.groupby('bairro', as_index=False)
    .agg(
        ocorrencias=('id', 'nunique'),
        mortos=('mortos', 'sum'),
        feridos=('feridos', 'sum'),
        baleados=('baleados', 'sum'),
        acoes_policiais=('acao_policial', 'sum'),
        chacinas=('chacina', 'sum'),
    )
)

seguranca_bairros = pd.DataFrame({'bairro': bairros_referencia})
seguranca_bairros = seguranca_bairros.merge(agg, on='bairro', how='left').fillna(0)

cols_int = ['ocorrencias', 'mortos', 'feridos', 'baleados', 'acoes_policiais', 'chacinas']
for col in cols_int:
    seguranca_bairros[col] = seguranca_bairros[col].astype(int)

seguranca_bairros['indice_risco'] = (
    seguranca_bairros['ocorrencias']
    + (2 * seguranca_bairros['mortos'])
    + seguranca_bairros['feridos']
    + (0.5 * seguranca_bairros['acoes_policiais'])
    + (3 * seguranca_bairros['chacinas'])
)

print(f'Bairros agregados: {len(seguranca_bairros)}')
display(seguranca_bairros.sort_values('indice_risco', ascending=False).head(15).reset_index(drop=True).style.hide(axis='index'))

## 2 - Top 10 bairros de maior e menor risco

In [ ]:
top10_risco = seguranca_bairros.nlargest(10, 'indice_risco').reset_index(drop=True)
top10_seguro = seguranca_bairros.nsmallest(10, 'indice_risco').reset_index(drop=True)

top10_risco.index += 1
top10_seguro.index += 1

print('Top 10 maior risco')
display(top10_risco.style.format({'indice_risco': '{:.1f}'}).background_gradient(subset=['indice_risco'], cmap='Reds').hide(axis='index'))

print('Top 10 menor risco')
display(top10_seguro.style.format({'indice_risco': '{:.1f}'}).background_gradient(subset=['indice_risco'], cmap='Greens_r').hide(axis='index'))

In [ ]:
fig_top = px.bar(
    top10_risco.sort_values('indice_risco'),
    x='indice_risco',
    y='bairro',
    orientation='h',
    color='indice_risco',
    color_continuous_scale='Reds',
    text='indice_risco',
    hover_data=['ocorrencias', 'mortos', 'feridos', 'baleados'],
    labels={'indice_risco': 'Índice de risco', 'bairro': 'Bairro'},
    title='Top 10 bairros com maior risco de segurança - Recife (2025)',
    height=500,
)
fig_top.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_top.update_layout(margin=dict(l=10, r=60, t=50, b=10), coloraxis_showscale=False)
fig_top.show()

## 3 - Mapa de hotspots

In [ ]:
mapa_df = df[(df['latitude'].notna()) & (df['longitude'].notna())].copy()
mapa_df = mapa_df[(mapa_df['latitude'] != 0) & (mapa_df['longitude'] != 0)]

center = [mapa_df['latitude'].mean(), mapa_df['longitude'].mean()]
mapa = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')

heat_data = mapa_df[['latitude', 'longitude', 'baleados']].copy()
heat_data['peso'] = heat_data['baleados'].clip(lower=1)

HeatMap(
    heat_data[['latitude', 'longitude', 'peso']].values.tolist(),
    radius=16,
    blur=22,
    min_opacity=0.25,
).add_to(mapa)

top_ocorrencias = mapa_df.sort_values(['baleados', 'mortos'], ascending=False).head(40)
for _, row in top_ocorrencias.iterrows():
    popup = (
        f"<strong>{row['bairro']}</strong><br>"
        f"{row['motivo_principal']}<br>"
        f"Mortos: {int(row['mortos'])} | Feridos: {int(row['feridos'])}<br>"
        f"{row['data_ocorrencia']}"
    )
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4 + min(int(row['baleados']), 8),
        color='#7f1d1d',
        fill=True,
        fill_color='#ef4444',
        fill_opacity=0.72,
        popup=folium.Popup(popup, max_width=320),
    ).add_to(mapa)

OUTPUT_MAPA.parent.mkdir(parents=True, exist_ok=True)
mapa.save(OUTPUT_MAPA)
print(f'Mapa salvo: {OUTPUT_MAPA}')
display(HTML(mapa._repr_html_()))

## 4 - Salvar agregado por bairro

In [ ]:
seguranca_bairros = seguranca_bairros.sort_values('indice_risco', ascending=False).reset_index(drop=True)
seguranca_bairros.to_csv(OUTPUT_BAIRROS, index=False, sep=CSV_SEPARATOR, encoding='utf-8')

print(f'Arquivo salvo: {OUTPUT_BAIRROS}')
print(f'Total de bairros: {len(seguranca_bairros)}')
print(f'Colunas: {seguranca_bairros.columns.tolist()}')
display(seguranca_bairros.head(10).style.format({'indice_risco': '{:.1f}'}).hide(axis='index'))

## 5 - Nota de 0 a 10 por bairro (normalização min-max invertida)

Como maior risco significa pior segurança, a nota é invertida:

$$\text{nota} = 10 - \frac{\text{risco} - \text{risco\_min}}{\text{risco\_max} - \text{risco\_min}} \times 10$$

Bairro com menor risco recebe nota **10,00** e o de maior risco recebe **0,00**.

In [ ]:
risco_min = seguranca_bairros['indice_risco'].min()
risco_max = seguranca_bairros['indice_risco'].max()

seguranca_bairros['nota_dimensao'] = 10 - (
    (seguranca_bairros['indice_risco'] - risco_min) / (risco_max - risco_min) * 10
)
seguranca_bairros['nota_dimensao'] = seguranca_bairros['nota_dimensao'].round(2)

print(f'Risco mínimo: {risco_min:.1f} -> nota 10,00')
print(f'Risco máximo: {risco_max:.1f} -> nota 0,00')
print(f'Risco médio : {seguranca_bairros["indice_risco"].mean():.1f}')

notas_exibir = (
    seguranca_bairros[['bairro', 'indice_risco', 'nota_dimensao', 'ocorrencias', 'mortos', 'feridos', 'baleados']]
    .sort_values('nota_dimensao', ascending=False)
    .reset_index(drop=True)
)
notas_exibir.index += 1

display(
    notas_exibir.style
    .format({'indice_risco': '{:.1f}', 'nota_dimensao': '{:.2f}'})
    .background_gradient(subset=['nota_dimensao'], cmap='RdYlGn')
    .set_caption('Nota de Segurança (0-10) por Bairro - Recife')
    .hide(axis='index')
)

In [ ]:
fig_notas = px.bar(
    notas_exibir.sort_values('nota_dimensao'),
    x='nota_dimensao',
    y='bairro',
    orientation='h',
    color='nota_dimensao',
    color_continuous_scale='RdYlGn',
    range_color=[0, 10],
    text='nota_dimensao',
    hover_data={'indice_risco': ':.1f', 'ocorrencias': True, 'baleados': True},
    labels={'nota_dimensao': 'Nota (0-10)', 'bairro': 'Bairro'},
    title='Índice de Segurança por Bairro - Recife (2025)',
    height=1800,
)
fig_notas.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_notas.update_layout(margin=dict(l=10, r=60, t=50, b=10), coloraxis_showscale=False, xaxis=dict(range=[0, 11]))
fig_notas.show()

## 6 - Salvar notas_seguranca.csv

In [ ]:
def formatar_dado(row):
    ocorrencias = int(row['ocorrencias'])
    baleados = int(row['baleados'])
    palavra_ocorrencia = 'ocorrencia' if ocorrencias == 1 else 'ocorrencias'
    palavra_baleado = 'baleado' if baleados == 1 else 'baleados'
    return f'{ocorrencias} {palavra_ocorrencia} e {baleados} {palavra_baleado}'


notas_seguranca = pd.DataFrame({
    'bairro': seguranca_bairros['bairro'],
    'nota_dimensao': seguranca_bairros['nota_dimensao'],
    'dado_principal': seguranca_bairros.apply(formatar_dado, axis=1),
})

notas_seguranca = notas_seguranca.sort_values('nota_dimensao', ascending=False).reset_index(drop=True)

notas_seguranca.to_csv(OUTPUT_NOTAS, index=False, sep=CSV_SEPARATOR, encoding='utf-8')

print(f'Arquivo salvo: {OUTPUT_NOTAS}')
print(f'Total de bairros: {len(notas_seguranca)}')
display(notas_seguranca.head(10).reset_index(drop=True).style.format(precision=2).hide(axis='index'))
print('...')
display(notas_seguranca.tail(10).reset_index(drop=True).style.format(precision=2).hide(axis='index'))

## 7 - Contexto oficial SDS-PE

In [ ]:
cvli = pd.read_excel(ROOT / 'data' / 'raw' / 'seguranca' / 'microdados_cvli.xlsx', sheet_name='Plan1')
cvp = pd.read_excel(ROOT / 'data' / 'raw' / 'seguranca' / 'microdados_cvp.xlsx', sheet_name='microdados cvp')

contexto_sds = pd.DataFrame([
    {
        'fonte': 'SDS-PE',
        'indicador': 'CVLI',
        'ano': YEAR,
        'municipio': 'Recife',
        'total': int(cvli[(cvli['MUNICIPIO'].str.upper() == 'RECIFE') & (cvli['ANO'] == YEAR)]['TOTAL DE VITIMAS'].sum()),
    },
    {
        'fonte': 'SDS-PE',
        'indicador': 'CVP',
        'ano': YEAR,
        'municipio': 'Recife',
        'total': int(cvp[(cvp['MUNICÍPIO'].str.upper() == 'RECIFE') & (cvp['ANO'] == YEAR)]['TOTAL'].sum()),
    },
])

contexto_sds.to_csv(OUTPUT_SDS, index=False, sep=CSV_SEPARATOR, encoding='utf-8')
print(f'Arquivo salvo: {OUTPUT_SDS}')
display(contexto_sds.style.hide(axis='index'))